<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 7 · Lunes — Introducción a Clasificación + KNN</h1>
<h3>De predecir números a predecir categorías</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

1. Entender la **diferencia** entre regresión y clasificación.
2. Conocer los **3 tipos de clasificación**: binaria, multiclase, multilabel.
3. Implementar el **clasificador KNN** y sus parámetros clave.
4. Visualizar las **fronteras de decisión** de un clasificador.
5. Resolver **2 ejercicios profundos** con datasets reales.

> Esta semana es completamente nueva: **dejamos de predecir números y pasamos a predecir categorías**.

# 1. Regresión vs Clasificación

Hasta ahora predecíamos **valores continuos** (precios, edades, puntajes). Ahora vamos a predecir **categorías** (clases).

| Aspecto | Regresión | Clasificación |
|---|---|---|
| Target | Número continuo (precio = 152,000 USD) | Categoría (spam / no spam) |
| Métrica típica | MAE, RMSE, R² | Accuracy, Precision, Recall, F1 |
| Algoritmos | LinearRegression, Ridge, RF Regressor | LogisticRegression, KNN, RF Classifier |
| Pregunta | ¿Cuánto? | ¿De qué clase? |

### Tipos de clasificación

| Tipo | Ejemplo | # de clases |
|---|---|---|
| **Binaria** | ¿Spam o no spam? | 2 |
| **Multiclase** | ¿Especie de pingüino? Adelie / Chinstrap / Gentoo | >2 (excluyentes) |
| **Multilabel** | Etiquetar fotos: gato Y árbol Y cielo | >1 etiqueta a la vez |

> 💡 Hoy nos enfocamos en **clasificación multiclase con Iris** y **binaria con Pima Diabetes**.

# 2. KNN Clasificador — "dime quién es tu vecino"

### 🧠 La idea

Para clasificar un punto nuevo, KNN:

1. Busca los **k vecinos más cercanos** del training set.
2. Cuenta de qué clase son.
3. **Vota por mayoría** → la clase más común es la predicción.

### Diferencia con KNN regresor

| Aspecto | KNN Regresor | KNN Clasificador |
|---|---|---|
| Output | Promedio de los k vecinos | Voto mayoritario |
| Método | `.predict()` da número | `.predict()` da clase + `.predict_proba()` da probabilidades |

### Parámetros clave

| Parámetro | ¿Qué hace? |
|---|---|
| `n_neighbors` (k) | Número de vecinos a consultar |
| `weights` | `'uniform'` (todos pesan igual) o `'distance'` (vecinos más cercanos pesan más) |
| `metric` | Métrica de distancia (`'euclidean'` por defecto) |

> ⚠️ **KNN exige estandarizar SIEMPRE** porque usa distancias.

## 👀 Demo en vivo — KNN sobre Iris (multiclase)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Dataset clásico de iris
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv'
cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df = pd.read_csv(url, names=cols)

print(f'Shape: {df.shape}')
print(f'Clases: {df["species"].value_counts().to_dict()}')
df.head()

In [ ]:
X = df.drop(columns=['species'])
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# stratify=y → mantiene la proporción de clases en train y test (CRÍTICO en clasificación)

sc = StandardScaler()
X_train_esc = sc.fit_transform(X_train)
X_test_esc  = sc.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_esc, y_train)

y_pred = knn.predict(X_test_esc)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'\nPredicciones primeras 5:')
for real, pred in zip(y_test[:5], y_pred[:5]):
    print(f'  Real: {real:15s} → Predicho: {pred}')

In [ ]:
# probabilidades por clase — algo que la regresión no tiene
probas = knn.predict_proba(X_test_esc[:5])
df_probas = pd.DataFrame(probas, columns=knn.classes_)
df_probas['Predicción'] = knn.predict(X_test_esc[:5])
df_probas['Real']       = y_test[:5].values
df_probas.round(2)

In [ ]:
# Visualizar las FRONTERAS DE DECISIÓN usando 2 features
# (esto es el superpoder visual de KNN — se ve cómo decide)
from matplotlib.colors import ListedColormap

X2 = df[['petal_length', 'petal_width']].values
y2 = pd.factorize(df['species'])[0]  # convertir clases a 0,1,2

knn2 = KNeighborsClassifier(n_neighbors=5).fit(X2, y2)

# Crear malla
x_min, x_max = X2[:,0].min()-1, X2[:,0].max()+1
y_min, y_max = X2[:,1].min()-0.5, X2[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.02))
Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FFAAAA','#AAFFAA','#AAAAFF']))
scatter = ax.scatter(X2[:,0], X2[:,1], c=y2, cmap=ListedColormap(['#FF0000','#00AA00','#0000FF']),
                      edgecolor='black', s=50)
ax.set_xlabel('petal_length')
ax.set_ylabel('petal_width')
ax.set_title('Fronteras de decisión del KNN (k=5) — cada color = una especie', fontweight='bold')
ax.legend(*scatter.legend_elements(), labels=list(df['species'].unique()))
plt.show()

## ⚙️ Tuning de k — ¿cuántos vecinos?

In [ ]:
ks = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]
scores_train, scores_test = [], []
for k in ks:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train_esc, y_train)
    scores_train.append(m.score(X_train_esc, y_train))
    scores_test.append(m.score(X_test_esc, y_test))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ks, scores_train, marker='o', linewidth=2, color='#70AD47', label='Train accuracy')
ax.plot(ks, scores_test, marker='o', linewidth=2, color='#C0504D', label='Test accuracy')
ax.set_xlabel('n_neighbors (k)'); ax.set_ylabel('Accuracy')
ax.set_title('Efecto de k — k=1 sobreajusta, k muy grande subajusta')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

print('👉 Con k=1 el train siempre da 1.0 (memoriza), pero test puede caer → overfitting.')
print('   Con k muy grande, ambos bajan → underfitting.')

---
# 🏋️ Ejercicios profundos

## 🌸 Ejercicio 1 — KNN multiclase con Iris (varias decisiones)

**Tarea integradora.** Quiero que vayas más allá del tutorial.

**Dataset:**
```python
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv'
cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df = pd.read_csv(url, names=cols)
```

**Parte A — Comparación con y SIN estandarización**
1. Entrenar 2 KNNs (k=5): uno con `StandardScaler`, otro sin escalar nada.
2. Reportar accuracy de cada uno. ¿Qué diferencia hay y por qué?

**Parte B — Búsqueda del mejor k**
3. Loop sobre k = [1, 3, 5, 7, 9, 11, 15, 21, 31].
4. Graficar accuracy de train Y test.
5. Encontrar el k óptimo y argumentarlo.

**Parte C — Efecto del peso de los vecinos**
6. Con el mejor k, probar `weights='uniform'` vs `weights='distance'`. ¿Cambia el resultado?

**Parte D — Predicción y probabilidades**
7. Predecir la especie de estas 2 flores nuevas:
   ```python
   flores = pd.DataFrame([
       {'sepal_length': 5.1, 'sepal_width': 3.5, 'petal_length': 1.4, 'petal_width': 0.2},
       {'sepal_length': 6.7, 'sepal_width': 3.0, 'petal_length': 5.2, 'petal_width': 2.3}
   ])
   ```
8. Mostrar las **probabilidades por clase** de cada predicción con `predict_proba()`.

**Parte E — Conclusión**
9. ¿Qué clase es más difícil de predecir? (pista: mira la matriz de confusión)
10. ¿Cuál sería tu modelo final y por qué?

In [ ]:
# Parte A — Con y sin estandarización 👇



In [ ]:
# Parte B + C — Tuning de k y de weights 👇



In [ ]:
# Parte D + E — Predicción + matriz de confusión + conclusión 👇



## 🩺 Ejercicio 2 — KNN binario con Pima Diabetes (clases desbalanceadas)

Trabajas en un hospital. Te dan datos de 768 mujeres del grupo Pima Indians con varias mediciones médicas y si tienen diabetes. Tu misión: predecir si una paciente nueva tiene diabetes.

**Dataset:**
```python
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']
df = pd.read_csv(url, names=cols)
```

**Target:** `Outcome` (0 = sana, 1 = diabetes). ⚠️ **Clases desbalanceadas:** ~65% sanas, ~35% con diabetes.

**Parte A — Exploración**
1. Cargar y revisar `.info()`, `.describe()`.
2. ¿Cuál es la proporción de clases?
3. **Truco oculto del dataset:** varias columnas tienen `0` que NO es un cero válido (ej. `Glucose=0` o `BMI=0` es imposible — son nulos disfrazados). Identificar cuáles y **reemplazarlos por NaN**.

**Parte B — Pipeline limpio**
4. Imputar los nulos (mediana) y estandarizar.
5. Train/test split 80/20 con `stratify=y` (mantiene la proporción de clases).

**Parte C — KNN + tuning**
6. Entrenar `KNeighborsClassifier` y hacer GridSearchCV sobre:
   - `n_neighbors`: [3, 5, 7, 11, 15, 21, 31]
   - `weights`: ['uniform', 'distance']
7. Reportar mejores hiperparámetros + accuracy en test.

**Parte D — Análisis crítico**
8. Calcular y mostrar la **matriz de confusión**.
9. ¿Cuántos **falsos negativos** hay? En medicina: un falso negativo = paciente con diabetes que el modelo dice estar sana → **NO RECIBE TRATAMIENTO**. ¿Qué tan grave es esto?
10. Si tuvieras que entregar este modelo a un hospital, ¿lo entregarías? ¿Qué mejorarías?

> 💡 Mañana veremos métricas que entienden el desbalance (precision, recall, F1, AUC). Hoy solo accuracy + matriz.

In [ ]:
# Parte A — Exploración + truco de los ceros 👇



In [ ]:
# Parte B + C — Pipeline + GridSearchCV 👇



In [ ]:
# Parte D — Matriz de confusión + análisis crítico 👇



---
## 📌 Cierre del día

- ✅ Diferencia clave: clasificación predice **categorías**, no números.
- ✅ KNN clasificador vota por mayoría entre los k vecinos más cercanos.
- ✅ Estandarizar es **obligatorio** (KNN usa distancias).
- ✅ `stratify=y` en el split mantiene la proporción de clases.
- ✅ `predict_proba()` da probabilidades por clase — útil en problemas donde no basta con la clase ganadora.
- ✅ Las **fronteras de decisión** son una forma visual potente de explicar cómo clasifica un modelo.

### 🔜 Mañana — Martes 2 de junio

Métricas de clasificación: **accuracy es engañosa** con clases desbalanceadas. Vamos a aprender precision, recall, F1, matriz de confusión y ROC-AUC.